# Final Evaluation — One-Week-Ahead Dengue Case Forecasting

This notebook performs the **single final evaluation** on the previously untouched 2021–2023 future holdout.

All development decisions are treated as frozen before this notebook is run.

## Forecasting scope

- **Outcome:** weekly dengue case count
- **Forecast horizon:** one week ahead
- **Prediction timing:** use information available through week `t-1` to predict week `t`
- **Weekly convention:** Sunday–Saturday, identified by Sunday `week_start_date`

## Frozen model specifications

1. **Persistence** — naïve benchmark: predict week `t` using `dengue_cases_lag1`
2. **SARIMA_1** — `(1,0,0) × (1,0,0,52)`
3. **XGB_3 history-only** — primary ML model
4. **XGB_3 history + climate** — direct climate-comparison model

## Final evaluation procedure

- Refit selected model specifications using all information available through **31 December 2020**
- Evaluate once on **2021–2023**
- Report:
  - MAE (primary metric)
  - RMSE
  - R²
  - higher-incidence-week error
  - actual-vs-predicted plots
- Do not alter model specifications after seeing final test performance

## 1. Setup and data loading

Input:

`data/processed/modelling/maynas_dengue_features_1w_ahead.csv`

All exported outputs use repository-relative paths only.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False

try:
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    STATSMODELS_AVAILABLE = True
except ImportError:
    STATSMODELS_AVAILABLE = False

warnings.filterwarnings(
    "ignore",
    message="Non-stationary starting autoregressive parameters"
)
warnings.filterwarnings(
    "ignore",
    message="Non-invertible starting MA parameters"
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 170)

PROJECT_ROOT = Path.cwd().resolve().parent

DATA_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "modelling"
    / "maynas_dengue_features_1w_ahead.csv"
)

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "final"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_FILE)

df["week_start_date"] = pd.to_datetime(
    df["week_start_date"],
    errors="raise"
)

df = (
    df
    .sort_values("week_start_date")
    .reset_index(drop=True)
)

print(f"Rows: {len(df):,}")
print(f"XGBoost available: {XGBOOST_AVAILABLE}")
print(f"Statsmodels available: {STATSMODELS_AVAILABLE}")

## 2. Freeze final train and test periods

All observed data through the end of 2020 are now available for refitting.

- **Refit period:** 2000–2020
- **Final test:** 2021–2023

In [ ]:
REFIT_END = pd.Timestamp("2020-12-31")
TEST_START = pd.Timestamp("2021-01-01")
TEST_END = pd.Timestamp("2023-12-31")

refit_df = df.loc[
    df["week_start_date"] <= REFIT_END
].copy()

test_df = df.loc[
    (df["week_start_date"] >= TEST_START)
    & (df["week_start_date"] <= TEST_END)
].copy()

split_summary = pd.DataFrame({
    "period": ["refit", "final_test"],
    "rows": [len(refit_df), len(test_df)],
    "first_week": [
        refit_df["week_start_date"].min(),
        test_df["week_start_date"].min(),
    ],
    "last_week": [
        refit_df["week_start_date"].max(),
        test_df["week_start_date"].max(),
    ],
    "mean_cases": [
        refit_df["dengue_cases"].mean(),
        test_df["dengue_cases"].mean(),
    ],
    "median_cases": [
        refit_df["dengue_cases"].median(),
        test_df["dengue_cases"].median(),
    ],
    "max_cases": [
        refit_df["dengue_cases"].max(),
        test_df["dengue_cases"].max(),
    ],
})

split_summary

### Interpretation — final holdout design

The final test period is now opened only after all model choices have been frozen.

No validation-driven tuning should occur from this point onward. The 2021–2023 results are therefore treated as the final estimate of out-of-sample performance under the selected forecasting design.

## 3. Metrics

In [ ]:
def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }

def clip_nonnegative(values):
    return np.clip(
        np.asarray(values, dtype=float),
        a_min=0,
        a_max=None,
    )

## 4. Final benchmark 1 — persistence

The persistence forecast remains:

`prediction(t) = dengue_cases_lag1`

In [ ]:
persistence_test = test_df[
    [
        "week_start_date",
        "dengue_cases",
        "dengue_cases_lag1",
    ]
].dropna().copy()

persistence_test["prediction"] = (
    persistence_test["dengue_cases_lag1"]
)

persistence_test["residual"] = (
    persistence_test["dengue_cases"]
    - persistence_test["prediction"]
)

persistence_metrics = regression_metrics(
    persistence_test["dengue_cases"],
    persistence_test["prediction"],
)

pd.Series(
    persistence_metrics,
    name="Persistence"
)

### Interpretation — persistence benchmark

This is the minimum standard the selected models should beat. Because recent dengue incidence is strongly autocorrelated, persistence is a meaningful rather than trivial comparator.

## 5. Final XGBoost feature specifications

In [ ]:
HISTORY_FEATURES = [
    "week_sin",
    "week_cos",
    "dengue_cases_lag1",
    "dengue_cases_lag2",
    "dengue_cases_lag4",
    "dengue_cases_lag8",
]

CLIMATE_FEATURES = [
    "precip_sum_mm_lag1",
    "precip_sum_mm_lag4",
    "precip_sum_mm_prev4w_sum",

    "temperature_c_mean_lag1",
    "temperature_c_mean_lag8",
    "temperature_c_mean_prev4w_mean",

    "specific_humidity_kgkg_mean_lag1",
    "relative_humidity_pct_mean_prev4w_mean",

    "wind_speed_ms_mean_lag1",
    "wind_speed_ms_mean_lag12",
    "wind_speed_ms_mean_prev4w_mean",

    "surface_pressure_hpa_mean_lag1",
    "surface_pressure_hpa_mean_lag8",
    "surface_pressure_hpa_mean_prev4w_mean",

    "ndvi_mean_weekly_lag1",
    "ndvi_mean_weekly_lag12",
    "ndvi_mean_weekly_prev4w_mean",
]

CLIMATE_ENHANCED_FEATURES = (
    HISTORY_FEATURES
    + CLIMATE_FEATURES
)

for feature in CLIMATE_ENHANCED_FEATURES:
    if feature not in df.columns:
        raise ValueError(
            f"Required frozen feature missing: {feature}"
        )

print(f"History-only features: {len(HISTORY_FEATURES)}")
print(
    f"History + climate features: "
    f"{len(CLIMATE_ENHANCED_FEATURES)}"
)

## 6. Final XGBoost model configuration

The frozen `XGB_3` configuration is:

- `n_estimators = 300`
- `max_depth = 2`
- `learning_rate = 0.05`
- `subsample = 1.0`
- `colsample_bytree = 1.0`
- `objective = reg:squarederror`

No further tuning is performed.

In [ ]:
if not XGBOOST_AVAILABLE:
    print(
        "XGBoost is not installed in this environment.\n"
        "Install it into dengue-venv with:\n\n"
        '& "$env:LOCALAPPDATA\\dengue-venv\\Scripts\\python.exe" '
        '-m pip install xgboost'
    )

In [ ]:
def make_xgb3():
    return XGBRegressor(
        objective="reg:squarederror",
        n_estimators=300,
        max_depth=2,
        learning_rate=0.05,
        subsample=1.0,
        colsample_bytree=1.0,
        random_state=42,
        n_jobs=-1,
    )

## 7. Final XGBoost — history only

In [ ]:
xgb_history_train = refit_df[
    ["week_start_date", "dengue_cases"]
    + HISTORY_FEATURES
].dropna().copy()

xgb_history_test = test_df[
    ["week_start_date", "dengue_cases"]
    + HISTORY_FEATURES
].dropna().copy()

xgb_history_model = make_xgb3()

xgb_history_model.fit(
    xgb_history_train[HISTORY_FEATURES],
    xgb_history_train["dengue_cases"],
)

xgb_history_test["prediction"] = (
    clip_nonnegative(
        xgb_history_model.predict(
            xgb_history_test[HISTORY_FEATURES]
        )
    )
)

xgb_history_test["residual"] = (
    xgb_history_test["dengue_cases"]
    - xgb_history_test["prediction"]
)

xgb_history_metrics = regression_metrics(
    xgb_history_test["dengue_cases"],
    xgb_history_test["prediction"],
)

pd.Series(
    xgb_history_metrics,
    name="XGB_3 history-only"
)

### Interpretation — history-only XGBoost

This is the primary machine-learning model selected during development. Its final test performance should be compared directly with persistence and SARIMA_1.

## 8. Final XGBoost — history + climate

In [ ]:
xgb_climate_train = refit_df[
    ["week_start_date", "dengue_cases"]
    + CLIMATE_ENHANCED_FEATURES
].dropna().copy()

xgb_climate_test = test_df[
    ["week_start_date", "dengue_cases"]
    + CLIMATE_ENHANCED_FEATURES
].dropna().copy()

xgb_climate_model = make_xgb3()

xgb_climate_model.fit(
    xgb_climate_train[
        CLIMATE_ENHANCED_FEATURES
    ],
    xgb_climate_train["dengue_cases"],
)

xgb_climate_test["prediction"] = (
    clip_nonnegative(
        xgb_climate_model.predict(
            xgb_climate_test[
                CLIMATE_ENHANCED_FEATURES
            ]
        )
    )
)

xgb_climate_test["residual"] = (
    xgb_climate_test["dengue_cases"]
    - xgb_climate_test["prediction"]
)

xgb_climate_metrics = regression_metrics(
    xgb_climate_test["dengue_cases"],
    xgb_climate_test["prediction"],
)

pd.Series(
    xgb_climate_metrics,
    name="XGB_3 history+climate"
)

### Interpretation — climate comparison

The direct comparison between the two frozen XGBoost specifications is the cleanest final test of whether environmental predictors improve one-week-ahead case forecasting beyond dengue history and seasonality.

## 9. Final SARIMA_1

The frozen traditional model is:

- non-seasonal order `(1, 0, 0)`
- seasonal order `(1, 0, 0, 52)`

The complete Sunday calendar is reconstructed so the four known historical dengue surveillance gaps remain missing rather than being treated as zero.

In [ ]:
calendar = pd.date_range(
    start=df["week_start_date"].min(),
    end=df["week_start_date"].max(),
    freq="W-SUN"
)

weekly = (
    df
    .set_index("week_start_date")
    .reindex(calendar)
)

weekly.index.name = "week_start_date"

sarima_refit_endog = (
    weekly.loc[
        weekly.index <= REFIT_END,
        "dengue_cases",
    ]
    .astype(float)
)

sarima_test_endog = (
    weekly.loc[
        (weekly.index >= TEST_START)
        & (weekly.index <= TEST_END),
        "dengue_cases",
    ]
    .astype(float)
)

print(
    f"SARIMA refit calendar weeks: "
    f"{len(sarima_refit_endog):,}"
)
print(
    f"SARIMA final test weeks: "
    f"{len(sarima_test_endog):,}"
)

In [ ]:
def rolling_one_step_forecast(
    fitted_result,
    test_endog,
):
    predictions = []
    result = fitted_result

    endog_name = test_endog.name

    for date in test_endog.index:

        forecast = result.get_forecast(
            steps=1
        )

        prediction = float(
            np.asarray(
                forecast.predicted_mean
            ).reshape(-1)[0]
        )

        predictions.append(
            {
                "week_start_date": date,
                "prediction_raw": prediction,
            }
        )

        actual = test_endog.loc[date]

        new_endog = pd.Series(
            [actual],
            index=pd.DatetimeIndex([date]),
            name=endog_name,
            dtype=float,
        )

        result = result.append(
            endog=new_endog,
            refit=False,
        )

    prediction_df = pd.DataFrame(
        predictions
    )

    prediction_df["prediction"] = (
        clip_nonnegative(
            prediction_df["prediction_raw"]
        )
    )

    return prediction_df

In [ ]:
if not STATSMODELS_AVAILABLE:
    print(
        "statsmodels is not installed in this environment.\n"
        "Install it into dengue-venv with:\n\n"
        '& "$env:LOCALAPPDATA\\dengue-venv\\Scripts\\python.exe" '
        '-m pip install statsmodels'
    )

In [ ]:
sarima_final_model = SARIMAX(
    sarima_refit_endog,
    order=(1, 0, 0),
    seasonal_order=(1, 0, 0, 52),
    trend="c",
    enforce_stationarity=False,
    enforce_invertibility=False,
)

sarima_final_fitted = (
    sarima_final_model.fit(
        disp=False,
        maxiter=200,
    )
)

sarima_test = rolling_one_step_forecast(
    sarima_final_fitted,
    sarima_test_endog,
)

sarima_test["dengue_cases"] = (
    sarima_test_endog.values
)

sarima_test["residual"] = (
    sarima_test["dengue_cases"]
    - sarima_test["prediction"]
)

sarima_metrics = regression_metrics(
    sarima_test["dengue_cases"],
    sarima_test["prediction"],
)

pd.Series(
    sarima_metrics,
    name="SARIMA_1"
)

### Interpretation — final SARIMA

SARIMA_1 is retained as the traditional time-series comparator selected during validation. Its final performance should not be used to revisit the frozen order specification.

## 10. Final model comparison

In [ ]:
final_comparison = pd.DataFrame([
    {
        "model": "Persistence",
        **persistence_metrics,
        "test_rows": len(persistence_test),
    },
    {
        "model": "SARIMA_1",
        **sarima_metrics,
        "test_rows": len(sarima_test),
    },
    {
        "model": "XGB_3 history-only",
        **xgb_history_metrics,
        "test_rows": len(xgb_history_test),
    },
    {
        "model": "XGB_3 history+climate",
        **xgb_climate_metrics,
        "test_rows": len(xgb_climate_test),
    },
])

final_comparison = (
    final_comparison
    .sort_values("MAE")
    .reset_index(drop=True)
)

final_comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(
    final_comparison["model"],
    final_comparison["MAE"]
)

ax.set_title(
    "Final Test MAE — 2021–2023"
)
ax.set_ylabel(
    "MAE (weekly dengue cases)"
)
ax.tick_params(
    axis="x",
    rotation=30
)

plt.tight_layout()
plt.show()

### Interpretation — final ranking

Use this table as the definitive model ranking. The lowest MAE is the primary criterion, with RMSE used to assess sensitivity to larger misses.

The final ranking should be reported even if it differs from the validation ranking.

## 11. Final climate contribution

Compare the frozen XGBoost history-only and history+climate models directly.

In [ ]:
climate_final_comparison = pd.DataFrame([
    {
        "comparison": "XGBoost climate increment",
        "history_MAE": xgb_history_metrics["MAE"],
        "climate_MAE": xgb_climate_metrics["MAE"],
        "MAE_change": (
            xgb_climate_metrics["MAE"]
            - xgb_history_metrics["MAE"]
        ),
        "MAE_pct_change": (
            (
                xgb_climate_metrics["MAE"]
                - xgb_history_metrics["MAE"]
            )
            / xgb_history_metrics["MAE"]
            * 100
        ),
        "history_RMSE": xgb_history_metrics["RMSE"],
        "climate_RMSE": xgb_climate_metrics["RMSE"],
        "RMSE_change": (
            xgb_climate_metrics["RMSE"]
            - xgb_history_metrics["RMSE"]
        ),
        "RMSE_pct_change": (
            (
                xgb_climate_metrics["RMSE"]
                - xgb_history_metrics["RMSE"]
            )
            / xgb_history_metrics["RMSE"]
            * 100
        ),
    }
])

climate_final_comparison

### Interpretation — H1

- Negative MAE/RMSE changes mean climate improved the frozen XGBoost model.
- Positive changes mean climate reduced final predictive accuracy.

The final conclusion for H1 should combine this overall result with the higher-incidence analysis below and the earlier development evidence.

## 12. Higher-incidence final-test performance

For consistency with development analysis, the full-series descriptive 95th-percentile threshold of **179.30 cases** is retained.

This remains an error-analysis threshold only, not an outbreak classification target.

In [ ]:
HIGH_INCIDENCE_THRESHOLD = 179.30

def incidence_error_summary(
    prediction_df,
    model_name,
):
    temp = prediction_df.copy()

    temp["absolute_error"] = (
        temp["dengue_cases"]
        - temp["prediction"]
    ).abs()

    temp["incidence_group"] = np.where(
        temp["dengue_cases"]
        >= HIGH_INCIDENCE_THRESHOLD,
        "Higher-incidence weeks",
        "Other weeks",
    )

    result = (
        temp
        .groupby(
            "incidence_group",
            as_index=False
        )
        .agg(
            weeks=("dengue_cases", "size"),
            mean_actual_cases=("dengue_cases", "mean"),
            MAE=("absolute_error", "mean"),
        )
    )

    result["model"] = model_name

    return result

final_incidence_errors = pd.concat(
    [
        incidence_error_summary(
            persistence_test,
            "Persistence",
        ),
        incidence_error_summary(
            sarima_test,
            "SARIMA_1",
        ),
        incidence_error_summary(
            xgb_history_test,
            "XGB_3 history-only",
        ),
        incidence_error_summary(
            xgb_climate_test,
            "XGB_3 history+climate",
        ),
    ],
    ignore_index=True,
)

final_incidence_errors[
    [
        "model",
        "incidence_group",
        "weeks",
        "mean_actual_cases",
        "MAE",
    ]
]

### Interpretation — high-incidence performance

This section determines whether the development-stage pattern persists: climate may be more useful during unusually high-incidence weeks even if it does not improve overall MAE.

Because the number of high-incidence test weeks may be small, interpret this descriptively rather than as a separate formal classification result.

## 13. Actual versus predicted

In [ ]:
plot_models = [
    (
        "Persistence",
        persistence_test,
    ),
    (
        "SARIMA_1",
        sarima_test,
    ),
    (
        "XGB_3 history-only",
        xgb_history_test,
    ),
    (
        "XGB_3 history+climate",
        xgb_climate_test,
    ),
]

for model_name, prediction_df in plot_models:

    fig, ax = plt.subplots(
        figsize=(14, 5)
    )

    ax.plot(
        prediction_df["week_start_date"],
        prediction_df["dengue_cases"],
        label="Actual",
    )

    ax.plot(
        prediction_df["week_start_date"],
        prediction_df["prediction"],
        label=model_name,
    )

    ax.set_title(
        f"Final Test Forecast — {model_name}"
    )
    ax.set_xlabel("Target week")
    ax.set_ylabel("Weekly dengue cases")
    ax.legend()

    plt.tight_layout()
    plt.show()

## 14. Final XGBoost feature importance

Feature importance is reported for the climate-enhanced final XGBoost model to describe which predictors the fitted model uses.

It should not be interpreted causally.

In [ ]:
xgb_final_importance = pd.DataFrame({
    "feature": CLIMATE_ENHANCED_FEATURES,
    "importance": (
        xgb_climate_model.feature_importances_
    ),
})

xgb_final_importance = (
    xgb_final_importance
    .sort_values(
        "importance",
        ascending=False,
    )
    .reset_index(drop=True)
)

xgb_final_importance.head(20)

In [ ]:
fig, ax = plt.subplots(
    figsize=(9, 7)
)

top_importance = (
    xgb_final_importance
    .head(15)
    .sort_values("importance")
)

ax.barh(
    top_importance["feature"],
    top_importance["importance"],
)

ax.set_title(
    "Final Climate-Enhanced XGBoost — Top 15 Feature Importances"
)
ax.set_xlabel("Importance")

plt.tight_layout()
plt.show()

### Interpretation — feature importance

Recent dengue incidence is expected to remain dominant. Environmental features with non-zero importance indicate that the model uses them to refine forecasts, but importance alone does not establish that they improve performance or represent causal drivers.

## 15. Save final evaluation outputs

These outputs represent the final, frozen holdout evaluation.

In [ ]:
final_comparison.to_csv(
    OUTPUT_DIR
    / "final_test_model_comparison.csv",
    index=False,
)

climate_final_comparison.to_csv(
    OUTPUT_DIR
    / "final_test_climate_increment.csv",
    index=False,
)

final_incidence_errors.to_csv(
    OUTPUT_DIR
    / "final_test_incidence_errors.csv",
    index=False,
)

xgb_final_importance.to_csv(
    OUTPUT_DIR
    / "final_xgb_climate_feature_importance.csv",
    index=False,
)

persistence_test[
    [
        "week_start_date",
        "dengue_cases",
        "prediction",
        "residual",
    ]
].to_csv(
    OUTPUT_DIR
    / "final_persistence_predictions.csv",
    index=False,
)

sarima_test[
    [
        "week_start_date",
        "dengue_cases",
        "prediction",
        "residual",
    ]
].to_csv(
    OUTPUT_DIR
    / "final_sarima_predictions.csv",
    index=False,
)

xgb_history_test[
    [
        "week_start_date",
        "dengue_cases",
        "prediction",
        "residual",
    ]
].to_csv(
    OUTPUT_DIR
    / "final_xgb_history_predictions.csv",
    index=False,
)

xgb_climate_test[
    [
        "week_start_date",
        "dengue_cases",
        "prediction",
        "residual",
    ]
].to_csv(
    OUTPUT_DIR
    / "final_xgb_climate_predictions.csv",
    index=False,
)

print(
    "Final holdout outputs saved to outputs/final/"
)

## 16. Final interpretation framework

After running the notebook, write the final conclusions without changing the models.

### H1 — climate/environmental contribution

Assess:

1. whether XGBoost history + climate beats XGBoost history-only on final MAE and RMSE;
2. whether climate changes error during higher-incidence weeks;
3. whether the final result is consistent with the mixed validation evidence from Random Forest, XGBoost and SARIMAX.

A valid conclusion may be **supported, partially supported, mixed, or not supported**.

### H2 — machine learning versus traditional time-series modelling

Compare:

- XGB_3 history-only
- SARIMA_1
- persistence

If XGBoost retains lower MAE/RMSE on the final future holdout, H2 receives final out-of-sample support.

### Final reporting principles

Report:

- the final test metrics exactly as observed;
- any change in ranking relative to validation;
- limitations caused by the small number of epidemic/high-incidence weeks;
- the strong role of recent dengue history;
- the model-dependent contribution of environmental predictors;
- that feature importance is predictive rather than causal.

Do not tune or change the models after this final evaluation.